In [0]:
!pip install langgraph
!pip install langchain
!pip install pygame
%pip install databricks-langchain
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import mlflow
import subprocess
import subprocess
import sys
from typing import Callable
import operator
from dataclasses import dataclass
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from databricks_langchain import ChatDatabricks
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.2,
    max_tokens=3000
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for endpoint in w.serving_endpoints.list():
    print(endpoint.name)

databricks-gpt-5-4
databricks-gpt-5-5
databricks-gpt-5-5-pro
databricks-gpt-5-4-mini
databricks-gpt-5-4-nano
databricks-gpt-5-2
databricks-gpt-oss-120b
databricks-gpt-5-3-codex
databricks-gpt-5-2-codex
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-gpt-5-1
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-meta-llama-3.1-405b-instruct


In [0]:
import os

print(os.getcwd())
OUTPUT_FILE = "/Workspace/Users/m.ahmadgill01@gmail.com/25280065_dino_runnerEC.py"
HIGH_SCORE_FILE = "/Workspace/Users/m.ahmadgill01@gmail.com/high_score.txt"

/Workspace/Users/m.ahmadgill01@gmail.com


##Task 1: Initializing System Memory and State Management [10 marks]

In [0]:

class GameState(TypedDict):
    # Stores all user/director prompts
    director_messages: Annotated[List[BaseMessage], add_messages]
    # Stores system design outputs from the architect
    architect_messages: Annotated[List[BaseMessage], add_messages]
    # Stores generated Python game code for each iteration
    engineer_code: Annotated[List[str], operator.add]
    # Stores QA analysis reports for each iteration
    qa_feedback: Annotated[List[str], operator.add]
    # Tracks which agent is currently active
    current_actor: str
    # Tracks iteration count of the system
    iteration: int
    # List of numeric scores from 1–10 per iteration
    iteration_score: Annotated[List[int], operator.add]
    # Boolean flag indicating whether code has been written to disk
    file_saved: bool


## Task 2: Implement Agent Nodes [40 marks]

### Subtask 2.1: Director and Architect Nodes

In [0]:
def director_node(state: GameState):
    director_msg = input("Awaiting Director Prompt: ")

    return {
        "director_messages": [HumanMessage(content=director_msg)],
        "current_actor": "architect",
        "iteration": state.get("iteration", 0) + 1,
    }


def architect_node(state: GameState):
    print(f"\n========== ITERATION {state.get('iteration', 0)} ==========")

    director_prompt = state["director_messages"][-1].content

    design_output = f"""
System Design for Dino Runner Game:

User Requirement:
{director_prompt}

Required Game Features:
1. Create a game window using Python and pygame.

2. Add a visually polished and more realistic dinosaur-style player character.
   The dinosaur must be created using pygame shapes, not external image assets.
   The dinosaur should include:
   - Body
   - Head
   - Eye
   - Mouth
   - Tail
   - Legs
   - Small arms
   - Ducking visual state

3. Add ground obstacles such as cactus-style obstacles.
   Cactus obstacles must look like cacti, not plain rectangles.
   They should use multiple pygame shapes such as rounded rectangles or small branches.

4. Add flying obstacles such as pterodactyl-style obstacles.
   Pterodactyl obstacles must look like flying creatures, not plain rectangles.
   They should include:
   - Body
   - Head
   - Beak
   - Wings
   - Eye

5. Implement accurate jumping and falling physics using velocity and gravity.

6. Implement correct user controls:
   - Space key for jump
   - Down arrow key for duck
   - Left/right arrow keys for optional movement

7. Jumping rules:
   - Only one jump is allowed at a time.
   - Double jump must not be allowed.
   - The dinosaur must land on the ground before another jump works.

8. Ducking rules:
   - Down arrow key should make the dinosaur duck.
   - Dinosaur should visually change while ducking.
   - Ducking should help avoid flying obstacles.

9. Add collision detection for both ground and flying obstacles.

10. Add a scoring system with these rules:
   - Score increases only while the game is active.
   - Score stops increasing after game over.
   - Final score remains visible on the game-over screen.
   - Score resets only when the player presses R to restart.

11. Add a high-score tracking system using a local text file.

12. High-score file path requirement:
   The generated code must work both in a normal local .py file and in a Jupyter/Databricks notebook.

   The code MUST include this exact high-score path logic:

   try:
       BASE_DIR = os.path.dirname(os.path.abspath(__file__))
   except NameError:
       BASE_DIR = os.getcwd()

   HIGH_SCORE_FILE = os.path.join(BASE_DIR, "high_score.txt")

   Rules:
   - Do not use only __file__ directly.
   - Do not use hardcoded Databricks paths.
   - Do not use absolute paths.
   - The high score must be loaded from high_score.txt.
   - If the file does not exist, start high score from 0.
   - If the file contains invalid data, start high score from 0.
   - Save the high score only when the final score is greater than the existing high score.

13. Obstacle spacing and lane safety rules:
   - Ground obstacles and flying obstacles must not spawn in the same vertical column or same x-position.
   - Do not create situations where a cactus and a flying obstacle block the player at the same time.
   - There must always be a clear possible path for the dinosaur.
   - Do not spawn a flying obstacle directly above a ground obstacle.
   - Do not spawn multiple obstacles together as an unavoidable wall.
   - Use one shared obstacle list instead of separate random spawning lists when possible.
   - Spawn only one obstacle at a time.
   - The next obstacle should spawn only after the previous obstacle has moved at least 550 pixels away.
   - Minimum obstacle spacing must be at least 550 pixels.
   - Flying obstacles should appear at a height where ducking can avoid them.
   - Ground obstacles should be avoidable by jumping.
   - The game must remain fair and playable.

14. Add game-over logic with:
   - R key to restart
   - Q key to quit

15. Add clean visuals:
   - Background
   - Ground line
   - Readable score
   - Readable high score
   - Realistic dinosaur shape
   - Realistic cactus shape
   - Realistic pterodactyl shape

16. Handle pygame.QUIT properly.

17. Keep the code clean, runnable, and simple.

Acceptance Criteria:
- The game process starts successfully.
- The code must run locally without manual changes.
- The code must also work in Jupyter/Databricks path handling.
- All required game features are implemented.
- Obstacles must not be too close.
- Ground and flying obstacles must not appear together in one blocking column.
- The game must always remain playable and fair.
- High score must persist in high_score.txt.
- Optional music or sound effects are not required.
- External image assets are not required.
- The code must be raw Python code only, without markdown fences.

Flying obstacle acceptance rule:
The game must visibly show pterodactyl/flying obstacles during normal gameplay. It is not enough to define the class. The code must spawn, move, draw, and check collision for flying obstacles. Use a forced alternating obstacle system if random spawning does not guarantee flying obstacles.
"""

    return {
        "architect_messages": [
            AIMessage(content=design_output)
        ],
        "current_actor": "engineer"
    }

### Subtask 2.2: Engineer Node

In [0]:
def engineer_node(state: GameState):
    architect_message = state["architect_messages"][-1].content if state.get("architect_messages") else ""

    prev_code = state["engineer_code"][-1] if state.get("engineer_code") else ""

    qa_messages = "\n".join(state.get("qa_feedback", []))

    engineer_prompt = f"""
You are an expert Python pygame developer.

Your task is to generate complete operational Python code based on the architect's design and QA feedback.

Architect Design:
{architect_message}

Previous Code:
{prev_code if prev_code else "No previous code exists."}

QA Feedback:
{qa_messages if qa_messages else "No QA feedback yet."}

Mandatory Implementation Requirements:
1. Return raw Python code only.
2. Do not include ```python or ``` markdown fences.
3. Use pygame.
4. The generated code must run locally and in Jupyter/Databricks without requiring manual changes.
5. The user should be able to run:
   python dino_runner.py
   without changing any path, filename, or code.

Path compatibility and high-score file requirement:
6. Do not use hardcoded Databricks paths.
7. Do not use absolute paths like /Workspace/Users/....
8. Use the same folder where the Python file is located.
9. The code MUST include this exact high-score path logic:

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HIGH_SCORE_FILE = os.path.join(BASE_DIR, "high_score.txt")

10. The game must automatically create or update high_score.txt in the same folder as the Python file.
11. The code must include all required imports, including os.
12. At game start, read the high score from this file if it exists.
13. If the file does not exist, safely start high score from 0.
14. If the file has invalid data, safely start high score from 0.
15. High score must be saved to the file only when a new high score is achieved.
16. Display the high score on the game screen.

Visual requirements:
17. Create a visually polished and more realistic dinosaur-style player character using pygame shapes.
18. Do not use external image assets.
19. The dinosaur must not be a plain rectangle.
20. The dinosaur should include:
   - Body
   - Head
   - Eye
   - Mouth
   - Tail
   - Legs
   - Small arms
   - Different ducking shape

Obstacle visual requirements:
21. Add ground obstacles such as cactus-style obstacles.
22. Cactus obstacles must look like cacti, not plain rectangles.
23. Add flying obstacles such as pterodactyl-style obstacles.
24. Pterodactyl obstacles must look like flying creatures, not plain rectangles.
25. Pterodactyl should include:
   - Body
   - Head
   - Beak
   - Wings
   - Eye
26. Use multiple pygame shapes such as circles, ellipses, polygons, and rounded rectangles.

Controls and movement:
27. Implement accurate jumping and falling physics using velocity and gravity.
28. Implement correct controls:
   - Space key = jump
   - Down arrow key = duck
   - Left/right arrow keys = optional movement

Jump rules:
29. Only one jump is allowed at a time.
30. Double jump must not be allowed.
31. The dinosaur must land on the ground before another jump works.
32. Use an on_ground or is_jumping flag to prevent double jump.

Ducking rules:
33. Down arrow key should make the dinosaur duck.
34. Dinosaur should visually change while ducking.
35. Ducking should help avoid flying obstacles.

Gameplay rules:
36. Add collision detection for both ground and flying obstacles.

Scoring and high-score rules:
37. Add a scoring system.
38. Score must increase only while the game is active and game_over is False.
39. When collision happens and game_over becomes True, the current score must stop increasing.
40. Do not reset the score immediately after game over.
41. Show the final score on the game-over screen.
42. Only reset score to 0 when the player presses R to restart.
43. High score must update only when current score is greater than saved high score.
44. High score must not increase automatically after game over.
45. High score must be saved to the file only when a new high score is achieved.
46. Display both current score and high score on the game screen.

Obstacle spacing and lane safety rules:
47. Obstacles must NOT spawn too close to each other.
48. There must be large visible space between obstacles.
49. Ground obstacles and flying obstacles must not spawn in the same vertical column or same x-position.
50. Do not create situations where a cactus and a flying obstacle block the player at the same time.
51. There must always be a clear possible path for the dinosaur.
52. Do not spawn a flying obstacle directly above a ground obstacle.
53. Do not spawn multiple obstacles together as an unavoidable wall.
54. Use one shared obstacle list instead of separate random spawning lists when possible.
55. Spawn only one obstacle at a time.
56. The next obstacle should spawn only after the previous obstacle has moved at least 550 pixels away.
57. Minimum obstacle spacing must be at least 550 pixels.
58. Flying obstacles should appear at a height where ducking can avoid them.
59. Ground obstacles should be avoidable by jumping.
60. Keep obstacle generation fair and playable.
61. Do not spawn many obstacles at the same time.
62. Ground and flying obstacles should appear in a balanced way.

Game-over requirement:
63. Add a game-over screen with:
   - R key = restart
   - Q key = quit

Other visual requirements:
64. Add clean visuals:
   - Background color
   - Ground line
   - Score text
   - High score text
   - Realistic dinosaur shape
   - Realistic cactus shape
   - Realistic pterodactyl shape

System requirement:
65. Handle pygame.QUIT properly.
66. Keep code clean, runnable, and simple.
67. If QA feedback exists, fix all issues mentioned.
  
Strict flying obstacle implementation requirement:
Flying obstacles are mandatory and must be visible during gameplay.

The code MUST include:
1. A Pterodactyl class.
2. A shared obstacles list containing both cactus and pterodactyl objects, OR a separate pterodactyls list that is actively used.
3. Actual pterodactyl spawning inside the main game loop.
4. A pterodactyl spawn condition that runs during gameplay.
5. Pterodactyl update/movement logic every frame.
6. Pterodactyl draw logic every frame.
7. Pterodactyl collision detection every frame.
8. Pterodactyl y-position must be above the ground, for example:
   y = GROUND_Y - 120
   or
   y = GROUND_Y - 150
9. Pterodactyl must move from right to left.
10. Pterodactyl must be avoidable by ducking.
11. Pterodactyl must not be only defined. It must appear in the running game.

Important:
Do not use only cactus obstacles.
Do not randomly choose cactus every time.
Use a forced obstacle type cycle if needed, for example:
- first obstacle = cactus
- second obstacle = pterodactyl
- third obstacle = cactus
- fourth obstacle = pterodactyl

This ensures flying obstacles definitely appear.
Pygame key constant requirement:
Use lowercase pygame key constants for letter keys.

Correct:
- pygame.K_r for restart
- pygame.K_q for quit

Incorrect:
- pygame.K_R
- pygame.K_Q

The generated code must never use pygame.K_R or pygame.K_Q because these do not exist and cause AttributeError.

Return only complete runnable Python code.
"""

    response = llm.invoke(engineer_prompt)

    generated_code = response.content

    return {
        "engineer_code": [generated_code],
        "current_actor": "execution_manager"
    }



## BonOus Part 

In [0]:
def syntax_checker_node(state: GameState):
    run_output = state["qa_feedback"][-1] if state.get("qa_feedback") else "No execution output found."
    engineer_code = state["engineer_code"][-1] if state.get("engineer_code") else "No engineer code found."

    syntax_prompt = f"""
You are a Syntax Checker.

Check the Python pygame code and execution output for real syntax/runtime problems only.

Execution Output:
{run_output}

Engineer Code:
{engineer_code}

Check only these real issues:
1. Python SyntaxError
2. ImportError
3. Traceback
4. Runtime exception
5. Invalid pygame key constants:
   - pygame.K_R is wrong
   - pygame.K_Q is wrong
   - pygame.K_r is correct
   - pygame.K_q is correct
6. Markdown fences inside code such as ```python or ```

Important:
- Ignore Pygame startup messages.
- Ignore XDG_RUNTIME_DIR warnings.
- Ignore ALSA audio warnings.
- These are environment warnings, not code errors.

Return a short syntax report.
"""

    response = llm.invoke(syntax_prompt)
    syntax_report = response.content

    print("\n===== SYNTAX CHECKER REPORT =====")
    print(syntax_report)

    return {
        "qa_feedback": [f"SYNTAX CHECKER REPORT:\n{syntax_report}"],
        "current_actor": "logic_tester"
    }
def logic_tester_node(state: GameState):
    design_requirements = state["architect_messages"][-1].content if state.get("architect_messages") else "No design requirements found."
    engineer_code = state["engineer_code"][-1] if state.get("engineer_code") else "No engineer code found."
    previous_qa = "\n\n".join(state.get("qa_feedback", []))

    logic_prompt = f"""
You are a Logic Tester.

Review the Python pygame code ONLY against the required design requirements.
Do not invent optional requirements.

Design Requirements:
{design_requirements}

Previous QA Reports:
{previous_qa}

Engineer Code:
{engineer_code}

Required Logic Checklist:
1. Does the code run or does the game process start successfully?
2. Is there a visually polished and more realistic dinosaur-style player?
3. Is the dinosaur made using pygame shapes instead of external assets?
4. Does the dinosaur include body, head, eye, mouth, tail, legs, and small arms?
5. Does the dinosaur have a different ducking visual state?
6. Are there ground obstacles such as cactus-style obstacles?
7. Do the cactus obstacles look like cacti instead of plain rectangles?
8. Are there flying obstacles such as pterodactyl-style obstacles?
9. Do the pterodactyl obstacles look like flying creatures instead of plain rectangles?
10. Do pterodactyls include body, head, beak, wings, and eye?
11. Are jumping and falling physics implemented using velocity and gravity?
12. Is only one jump allowed at a time?
13. Is double jump prevented using on_ground or is_jumping logic?
14. Are controls correctly implemented?
   - Space = jump
   - Down arrow = duck
   - Left/right arrows = optional movement
15. Does ducking visually change the dinosaur?
16. Does ducking help avoid flying obstacles?
17. Is collision detection implemented for both ground and flying obstacles?

Scoring Logic Checklist:
18. Is scoring implemented?
19. Does score increase only when game_over is False?
20. Does score stop increasing after collision/game over?
21. Does final score remain visible on the game-over screen?
22. Does score reset only when the player presses R to restart?
23. Does high score update only if current score is greater than saved high score?
24. Does high score avoid increasing automatically after game over?
25. Is high score displayed on screen?

High-score File Checklist:
26. Does the code avoid hardcoded Databricks paths?
27. Does the code avoid absolute paths like /Workspace/Users/...?
28. Does the code include this exact safe path logic?

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HIGH_SCORE_FILE = os.path.join(BASE_DIR, "high_score.txt")

29. Is high score stored in high_score.txt in the same folder as the Python file or current notebook directory?
30. Does the code read high score from the file at game start?
31. Does the code handle missing high-score file safely?
32. Does the code handle invalid high-score file data safely?
33. Does the code write/update the file only when a new high score is achieved?

Game-over Checklist:
34. Is game-over logic implemented?
35. Are restart and quit controls implemented?
   - R = restart
   - Q = quit
36. Is pygame.QUIT handled properly?
37. Are visuals clean and readable?

Pygame Key Validation:
- Check that the code uses pygame.K_r, not pygame.K_R.
- Check that the code uses pygame.K_q, not pygame.K_Q.
- If pygame.K_R or pygame.K_Q is used, mark it as a real runtime bug.

Strict Flying Obstacle QA:
Fail the flying obstacle requirement if:
1. There is no Pterodactyl class or flying obstacle class.
2. The class exists but is never instantiated.
3. The pterodactyl/flying obstacle is not appended to an active obstacle list.
4. The pterodactyl/flying obstacle is not updated every frame.
5. The pterodactyl/flying obstacle is not drawn every frame.
6. The pterodactyl/flying obstacle has no collision detection.
7. The pterodactyl/flying obstacle appears on the ground instead of above ground.

Pass only if flying obstacles actually appear during gameplay.

Return a short logic testing report with:
- Passed requirements
- Failed requirements
- Real bugs only
"""

    response = llm.invoke(logic_prompt)
    logic_report = response.content

    print("\n===== LOGIC TESTER REPORT =====")
    print(logic_report)

    return {
        "qa_feedback": [f"LOGIC TESTER REPORT:\n{logic_report}"],
        "current_actor": "performance_auditor"
    }
def performance_auditor_node(state: GameState):
    engineer_code = state["engineer_code"][-1] if state.get("engineer_code") else "No engineer code found."
    previous_qa = "\n\n".join(state.get("qa_feedback", []))

    performance_prompt = f"""
You are a Performance Auditor and QA Summary Scorer.

Review the code quality, game fairness, and obstacle spacing.

Previous QA Reports:
{previous_qa}

Engineer Code:
{engineer_code}

Performance and Fairness Checklist:
1. Are obstacles prevented from spawning too close to each other?
2. Is there large visible space between obstacles?
3. Is there a spawn timer, minimum distance, or cooldown system?
4. Is minimum obstacle spacing at least 550 pixels?
5. Does the code avoid spawning too many obstacles at once?
6. Does the code avoid spawning ground and flying obstacles in the same vertical column or same x-position?
7. Does the code avoid placing a flying obstacle directly above a ground obstacle?
8. Does the code avoid creating an unavoidable wall?
9. Does the code spawn only one obstacle at a time or otherwise guarantee a clear possible path?
10. Are ground obstacles avoidable by jumping?
11. Are flying obstacles avoidable by ducking?
12. Is the game loop simple and readable?
13. Is pygame.QUIT handled properly?

Important:
- Ignore music, sound effects, images, and advanced assets.
- Ignore XDG_RUNTIME_DIR and ALSA warnings.
- Do not invent optional requirements.

Now combine all QA results into one final QA report.

Return a short final QA report with:
- Syntax checker summary
- Logic tester summary
- Performance auditor summary
- Passed requirements
- Failed requirements
- Real bugs only
- Final QA summary
"""

    response = llm.invoke(performance_prompt)
    final_qa_report = response.content

    print("\n===== PERFORMANCE AUDITOR / FINAL QA REPORT =====")
    print(final_qa_report)

    return {
        "qa_feedback": [final_qa_report],
        "current_actor": "scorer"
    }

### Subtask 2.3: File I/O and Execution Nodes

In [0]:
def clean_code_output(code: str) -> str:
    code = code.strip()

    if code.startswith("```python"):
        code = code.replace("```python", "", 1).strip()

    elif code.startswith("```"):
        code = code.replace("```", "", 1).strip()

    if code.endswith("```"):
        code = code[:-3].strip()

    return code

In [0]:
def file_writer(state: GameState):
    code = state["engineer_code"][-1]
    code = clean_code_output(code)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(code)
    print(code[:500])
    print(f"\n[File saved: {OUTPUT_FILE}]")
    return {
        "file_saved": True,
        "current_actor": "run_code"
    }
def run_code(state: GameState):
    print("\n===== CODE EXECUTION =====")
    choice = input("Run the generated game? (y/n): ").strip().lower()

    if choice != "y":
        return {
            "qa_feedback": ["Execution skipped by human."],
            "current_actor": "qa_engineer"
        }

    process = subprocess.Popen(
        [sys.executable, "dino_runner.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    execution_output = f"""
Game process started successfully.

Process ID:
{process.pid}

Note:
The game is running in a separate process.
Close the game manually when done.
"""

    print(execution_output)

    return {
        "qa_feedback": [execution_output],
        "current_actor": "qa_engineer"
    }

### Subtask 2.4: QA and Scorer Nodes

In [0]:



def score_node(state: GameState):
    qa_feedback = state["qa_feedback"][-1] if state.get("qa_feedback") else "No QA feedback found."

    score_prompt = f"""
You are a fair Scorer.

Read the QA feedback and assign a score from 1 to 10.

Score based ONLY on required features, not optional polish.

Required core features:
1. Code runs or game process starts successfully
2. Visually polished and more realistic dinosaur player
3. Ground obstacles/cacti
4. Flying obstacles/pterodactyls
5. Jumping and falling physics
6. Only one jump allowed at a time
7. Ducking mechanics
8. Correct keyboard controls
9. Collision detection
10. Score system
11. Score stops after game over
12. Final score stays visible after game over
13. High-score tracking using high_score.txt
14. Safe path logic for both local .py and Jupyter/Databricks
15. Obstacles are not too close
16. Ground and flying obstacles do not form an unavoidable wall
17. Game-over screen
18. Restart and quit controls
19. pygame.QUIT handling

Scoring rules:
1-3 = Code has syntax errors or cannot start
4-6 = Code starts but misses many required features
7 = Code works partially but misses important required features
8 = Code works and includes most required features
9 = Code works and includes all required features with only minor issues
10 = Complete, polished, clean, and includes all required features

Important:
- Do not reduce score for missing music, sound effects, external images, or advanced assets.
- Do not reduce score for XDG_RUNTIME_DIR or ALSA warnings.
- These warnings are caused by the Databricks/Linux environment, not the game code.
- If the game process started successfully and all required features are present, score should be 9 or 10.
- If QA only mentions optional improvements, score should be 9 or 10.
- Return only one number from 1 to 10.

QA Feedback:
{qa_feedback}
"""

    response = llm.invoke(score_prompt)

    score_text = response.content.strip()

    try:
        score = int(score_text)
    except:
        score = 8

    score = max(1, min(score, 10))

    print("\n===== ITERATION SCORE =====")
    print(score)

    return {
        "iteration_score": [score],
        "current_actor": "human_review"
    }

In [0]:
def should_continue(state: GameState):
    latest_score = state["iteration_score"][-1] if state.get("iteration_score") else 0

    if latest_score >= 8:
        print("\nScore is good. Finalizing project.")
        return "finish"
    else:
        print("\nScore is low. Sending back to Engineer for improvement.")
        return "engineer"

## Task 3: Create Graph Structure [20 marks]

# Bonous Part

In [0]:
# -----------------------------
# Build QA Subgraph first
# -----------------------------

qa_builder = StateGraph(GameState)

qa_builder.add_node("syntax_checker", syntax_checker_node)
qa_builder.add_node("logic_tester", logic_tester_node)
qa_builder.add_node("performance_auditor", performance_auditor_node)

qa_builder.set_entry_point("syntax_checker")

qa_builder.add_edge("syntax_checker", "logic_tester")
qa_builder.add_edge("logic_tester", "performance_auditor")
qa_builder.add_edge("performance_auditor", END)

qa_subgraph = qa_builder.compile()


# -----------------------------
# Build Parent Graph
# -----------------------------

graph_builder = StateGraph(GameState)

graph_builder.add_node("director", director_node)
graph_builder.add_node("architect", architect_node)
graph_builder.add_node("engineer", engineer_node)
graph_builder.add_node("file_writer", file_writer)
graph_builder.add_node("run_code", run_code)

# Bonus Part:
# QA_Subgraph appears as one node in the parent graph
graph_builder.add_node("qa_engineer", qa_subgraph)

graph_builder.add_node("scorer", score_node)

graph_builder.set_entry_point("director")

graph_builder.add_edge("director", "architect")
graph_builder.add_edge("architect", "engineer")
graph_builder.add_edge("engineer", "file_writer")
graph_builder.add_edge("file_writer", "run_code")
graph_builder.add_edge("run_code", "qa_engineer")
graph_builder.add_edge("qa_engineer", "scorer")

graph_builder.add_conditional_edges(
    "scorer",
    should_continue,
    {
        "engineer": "engineer",
        "finish": END
    }
)

game_graph = graph_builder.compile()

## Task 4: System Invocation [10 marks]

In [0]:
config = {
    "configurable": {
        "thread_id": "dino_runner_session_1"
    },
    "recursion_limit": 20
}
memory = MemorySaver()
initial_state: GameState = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": [],
    "qa_feedback": [],
    "current_actor": "director",
    "iteration": 0,
    "iteration_score": [],
    "file_saved": False,
}
print("--- STARTING WORKFLOW ---")

for event in game_graph.stream(initial_state, config=config):
    for node, value in event.items():
        print(f"\n===== {node.upper()} =====")

        if isinstance(value, dict):
            for key, val in value.items():
                print(f"{key}: {val}")
        else:
            print(value)

print("\n--- WORKFLOW FINISHED ---")
        



--- STARTING WORKFLOW ---


Awaiting Director Prompt:  Build a Dino Runner game


===== DIRECTOR =====
director_messages: [HumanMessage(content='Build a Dino Runner game', additional_kwargs={}, response_metadata={}, id='c163f325-3ad2-42c5-abdf-240dcbaf0069')]
current_actor: architect
iteration: 1

========== ITERATION 1 ==========

===== ARCHITECT =====
architect_messages: [AIMessage(content='\nSystem Design for Dino Runner Game:\n\nUser Requirement:\nBuild a Dino Runner game\n\nRequired Game Features:\n1. Create a game window using Python and pygame.\n\n2. Add a visually polished and more realistic dinosaur-style player character.\n   The dinosaur must be created using pygame shapes, not external image assets.\n   The dinosaur should include:\n   - Body\n   - Head\n   - Eye\n   - Mouth\n   - Tail\n   - Legs\n   - Small arms\n   - Ducking visual state\n\n3. Add ground obstacles such as cactus-style obstacles.\n   Cactus obstacles must look like cacti, not plain rectangles.\n   They should use multiple pygame shapes such as rounded rectangles or small branches.\n\n4

Run the generated game? (y/n):  y


Game process started successfully.

Process ID:
3652

Note:
The game is running in a separate process.
Close the game manually when done.


===== RUN_CODE =====
qa_feedback: ['\nGame process started successfully.\n\nProcess ID:\n3652\n\nNote:\nThe game is running in a separate process.\nClose the game manually when done.\n']
current_actor: qa_engineer

===== SYNTAX CHECKER REPORT =====
Syntax Report:

* No Python SyntaxError found.
* No ImportError found.
* No Traceback found.
* No Runtime exception found.
* Invalid pygame key constants: 
  - `pygame.K_R` should be `pygame.K_r` 
  - `pygame.K_Q` should be `pygame.K_q`
* No Markdown fences inside code found.

The code provided seems to be syntactically correct, but there are two invalid pygame key constants that should be corrected to `pygame.K_r` and `pygame.K_q`.

===== LOGIC TESTER REPORT =====
Logic Testing Report:

Passed Requirements:
1. The code runs or the game process starts successfully.
2. There is a visually polished and mo